# 1. EDA — Biohub Cell Tracking During Development

Purpose: understand the OME-Zarr + GEFF data format, survey shape/scale/
ground-truth density across multiple training videos (not just one), and
inspect a single video in depth (frame visualization, division timing)
before designing/tuning a model.

Runs on Kaggle via `scripts/push_kaggle_kernel.sh eda` (the competition
mount and the public `cellmot-baseline-artifacts` dataset, which bundles a
working `repo/`, both auto-detect — see the Setup cell), or locally after
downloading one sample and pointing `$CELLMOT_DATA_DIR` at it. Verified
working end-to-end on Kaggle (kernel `tuannm3812/biohub-eda`) — results
below and in `docs/2_eda_insights.md`.

Project docs (strategy, experiment log, submission history):
[`docs/`](https://github.com/tuannm3812/kaggle-biohub-cell-tracking-during-development/tree/main/docs).

## 1. Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle").exists()


def _find_mount(candidates: list[Path], marker: str) -> Path | None:
    """Return the first candidate containing ``marker``, else scan /kaggle/input."""
    for c in candidates:
        if (c / marker).exists():
            return c
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for p in kaggle_input.glob(f"**/{marker}"):
            return p.parent
    return None


if IS_KAGGLE:
    # Pin numpy/scipy/torch to whatever Kaggle's base image already has:
    # letting pip pick a newer numpy for zarr>=3.0.10 breaks the base
    # image's precompiled scipy (ImportError deep in scipy.spatial/
    # numpy._core -- an ABI mismatch between new numpy and the untouched
    # old scipy build). tracksdata also depends on torch; pin it too so an
    # unpinned reinstall can't swap the base image's build for one that
    # doesn't match its GPU driver (irrelevant with GPU off here, but keeps
    # this cell identical to 02_baseline_modeling.ipynb's). Both issues
    # observed directly on this competition's Kaggle image. Must also run
    # before importing numpy/matplotlib below, for the same reason.
    import importlib.metadata

    _pinned = {
        pkg: importlib.metadata.version(pkg) for pkg in ("numpy", "scipy", "torch")
    }
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            *(f"{pkg}=={ver}" for pkg, ver in _pinned.items()),
            "zarr>=3.0.10", "tqdm", "polars", "imageio",
            "tracksdata @ git+https://github.com/royerlab/tracksdata@main",
        ],
        check=True,
    )
    ARTIFACTS_MOUNT = _find_mount(
        [
            Path("/kaggle/input/cellmot-baseline-artifacts"),
            Path("/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
        ],
        "weights",
    )
    if ARTIFACTS_MOUNT is None:
        raise FileNotFoundError(
            "cellmot-baseline-artifacts dataset not found under /kaggle/input -- add "
            "it as a data source (see kernel-metadata.json)."
        )
    REPO_ROOT = ARTIFACTS_MOUNT / "repo"  # read-only is fine -- EDA never writes predictions/weights
else:
    REPO_ROOT = Path.cwd().parent

import matplotlib.pyplot as plt  # noqa: E402 -- see the pip-install note above
import numpy as np  # noqa: E402

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "scripts"))

from dataspec import DATASET_PATH  # noqa: E402 -- needs REPO_ROOT on sys.path first

from tracking_cellmot.io import list_datasets, open_dataset  # noqa: E402

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR = Path(f"/kaggle/input/competitions/{COMPETITION}")
TEST_DIR = COMP_DIR / "test" if IS_KAGGLE else REPO_ROOT / "data" / "test"

# Tables/plots render on the Kaggle kernel page but aren't captured by the
# CLI-downloadable log -- save them as files under Output instead, so they
# can be pulled with `kaggle kernels output` and embedded in
# docs/2_eda_insights.md.
OUTPUT_DIR = Path("/kaggle/working") if IS_KAGGLE else REPO_ROOT / "notebooks" / "eda_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 0
np.random.seed(SEED)

DATASET_NAME = None      # None -> pick the first available training dataset
FRAME_INDEX = 0           # timepoint to visualize for the single-dataset deep dive
N_VIDEOS_FOR_STATS = 10   # how many train videos to summarize in the stats table; None = all
N_GIF_FRAMES = 30         # timepoints to render in the animated track-preview GIF

print(f"IS_KAGGLE={IS_KAGGLE}  REPO_ROOT={REPO_ROOT}  DATASET_PATH={DATASET_PATH}  TEST_DIR={TEST_DIR}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")

## 2. List available datasets (train + test)

In [ ]:
train_datasets = list_datasets(DATASET_PATH, require_geff=True)
print(f"DATASET_PATH = {DATASET_PATH}")
print(f"{len(train_datasets)} train datasets with ground truth found")
for p in train_datasets[:10]:
    print(" -", p.stem)

test_datasets = list_datasets(TEST_DIR, require_geff=False) if TEST_DIR.exists() else []
print(f"\nTEST_DIR = {TEST_DIR}")
print(f"{len(test_datasets)} test datasets found (no ground truth, per the competition)")
for p in test_datasets[:10]:
    print(" -", p.stem)

## 3. Multi-video stats table (train)

A single video isn't representative — summarize shape, scale, and
ground-truth density across the first `N_VIDEOS_FOR_STATS` train videos so
modeling decisions (e.g. `--unet-batch-size`, `--det-threshold` in
`02_baseline_modeling.ipynb`) are grounded across the dataset, not one clip.

In [ ]:
import polars as pl

survey = train_datasets[:N_VIDEOS_FOR_STATS] if N_VIDEOS_FOR_STATS else train_datasets

rows = []
for p in survey:
    ds_i = open_dataset(p, normalize=False, require_tracks=True)
    out_degree_i = ds_i.tracks.edge_attrs().group_by("source_id").len()
    n_t = ds_i.image.shape[0]
    n_nodes = ds_i.tracks.num_nodes()
    scale_z, scale_y, scale_x = (round(s, 4) for s in ds_i.scale)
    rows.append({
        "dataset": p.stem,
        "T": n_t,
        "Z": ds_i.image.shape[1],
        "Y": ds_i.image.shape[2],
        "X": ds_i.image.shape[3],
        "dtype": str(ds_i.image.dtype),
        "scale_z_um": scale_z,
        "scale_y_um": scale_y,
        "scale_x_um": scale_x,
        "nodes": n_nodes,
        "edges": ds_i.tracks.num_edges(),
        "divisions": int((out_degree_i["len"] == 2).sum()),
        "nodes_per_timepoint": round(n_nodes / n_t, 2),
    })

stats = pl.DataFrame(rows)
stats.write_csv(OUTPUT_DIR / "stats.csv")  # flat scalar columns only -- CSV can't hold the old tuple scale column
print(f"Surveyed {len(survey)}/{len(train_datasets)} train videos -- saved to {OUTPUT_DIR / 'stats.csv'}")
stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(stats["dataset"], stats["nodes_per_timepoint"], color=plt.cm.viridis(0.5))
axes[0].set_title("Annotated nodes per timepoint")
axes[0].tick_params(axis="x", rotation=90)
axes[1].bar(stats["dataset"], stats["divisions"], color=plt.cm.viridis(0.8))
axes[1].set_title("Division events")
axes[1].tick_params(axis="x", rotation=90)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "stats_bars.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {OUTPUT_DIR / 'stats_bars.png'}")

*Insight: all 10 surveyed videos share identical shape/scale —
`(100, 64, 256, 256)`, `(1.625, 0.4062, 0.4062)` µm — so a single global
`--det-threshold` is reasonable, at least for this sample. Annotation
density itself varies ~15× (0.51 to 7.88 nodes/timepoint) with no visible
link to video shape. Full data in `docs/2_eda_insights.md`.*

## 4. Count calibration signal: `estimated_number_of_nodes`

Per `docs/3_strategy.md` (synthesized from public reference notebooks):
each `.geff`'s `zarr.json` metadata carries
`attributes.geff.extra.estimated_number_of_nodes` — an estimate of the
*true* cell count, independent of the sparse GT annotation count above.
Several top public submissions use the ratio of this estimate to their own
detection count to calibrate a per-video detection budget (avoiding the
Adjusted Edge Jaccard's over-prediction penalty — `docs/metrics.md`). Not
yet used anywhere in our own pipeline; this cell just measures it.

In [ ]:
import json as _json


def read_estimated_nodes(geff_path: Path) -> int | None:
    """Read attributes.geff.extra.estimated_number_of_nodes from a .geff's zarr.json, if present."""
    meta_path = geff_path.with_suffix(".geff") / "zarr.json"
    if not meta_path.exists():
        return None
    try:
        meta = _json.loads(meta_path.read_text())
    except (OSError, ValueError):
        return None
    return meta.get("attributes", {}).get("geff", {}).get("extra", {}).get("estimated_number_of_nodes")


calib_rows = []
for p, row in zip(survey, stats.iter_rows(named=True), strict=True):
    est = read_estimated_nodes(p)
    calib_rows.append({
        "dataset": row["dataset"],
        "annotated_nodes": row["nodes"],
        "estimated_number_of_nodes": est,
        "annotated_fraction": round(row["nodes"] / est, 4) if est else None,
    })

calib = pl.DataFrame(calib_rows)
calib.write_csv(OUTPUT_DIR / "calib.csv")
n_with_estimate = calib["estimated_number_of_nodes"].is_not_null().sum()
print(f"{n_with_estimate}/{len(calib)} surveyed videos have an estimated_number_of_nodes field")
print(f"Saved {OUTPUT_DIR / 'calib.csv'}")
calib

*Insight: `estimated_number_of_nodes` is present on **10/10** surveyed
train videos — the field exists and is usable. `annotated_fraction`
(annotated / estimated true count) ranges **0.13%–1.34%** across the 10
videos — a much stronger sparsity statement than the raw node counts
alone: true cell counts are 15,000–79,000 per video, and only
dozens-to-hundreds are annotated. Full table in `docs/2_eda_insights.md`.
Whether this field is **also** present on the real `test/` videos is
checked next, below.*

### Also check `estimated_number_of_nodes` on the real `test/` videos

The check above only covers **train** videos, which have a `.geff` because
they carry ground truth. The `DET_THRESHOLD=0.90` sweep in
`02_baseline_modeling.ipynb` predicted a gain on a 19-video train fold but
regressed sharply on the real (4-video) test set — one open hypothesis is
that a count-budget calibration built from train videos doesn't transfer
to test (`docs/3_strategy.md`, `docs/4_experiments.md`). That only matters
if this field is even readable on test videos, which don't ship GT — this
cell checks directly, reusing `read_estimated_nodes` unchanged against
`test_datasets` (section 2).

In [ ]:
test_calib_rows = []
for p in test_datasets:
    est = read_estimated_nodes(p)
    test_calib_rows.append({"dataset": p.stem, "estimated_number_of_nodes": est})

test_calib = pl.DataFrame(test_calib_rows)
test_calib.write_csv(OUTPUT_DIR / "test_calib.csv")
n_test_with_estimate = test_calib["estimated_number_of_nodes"].is_not_null().sum() if test_calib.height else 0
print(f"{n_test_with_estimate}/{len(test_calib)} test videos have an estimated_number_of_nodes field")
print(f"Saved {OUTPUT_DIR / 'test_calib.csv'}")
test_calib

*Insight: **0/4 test videos have `estimated_number_of_nodes`** — confirmed
on the real competition test set (kernel `tuannm3812/biohub-eda` v8,
2026-07-22). Not partially there, not present under a different key: the
test videos ship no `.geff` at all, so there's no `zarr.json` to read this
metadata from. This directly rules out the "calibrate `DET_THRESHOLD` per
test video against its true-count estimate" approach considered in
`docs/3_strategy.md` — the signal it depends on isn't available at
inference time. See `docs/2_eda_insights.md` and `docs/3_strategy.md`.*

## 5. Deep dive: one dataset in detail

In [ ]:
name = DATASET_NAME or train_datasets[0].stem
ds = open_dataset(DATASET_PATH / name, normalize=False, require_tracks=True)

print(f"dataset:      {name}")
print(f"image shape:  {ds.image.shape}  (T, Z, Y, X)")
print(f"image dtype:  {ds.image.dtype}")
print(f"voxel scale:  {ds.scale}  microns (Z, Y, X)")
print(f"track nodes:  {ds.tracks.num_nodes()}")
print(f"track edges:  {ds.tracks.num_edges()}")

*Insight: `44b6_0113de3b` has 100 timepoints but only 52 annotated nodes /
50 edges total — ~0.5 nodes per timepoint on a `(64, 256, 256)` volume.
Confirms the GT is genuinely sparse, matching the competition's own
framing. Per section 3/4's wider 10-video survey, this video sits at the
**low end** of the annotation-density range (0.51–7.88 nodes/timepoint) —
see `docs/2_eda_insights.md` for the full writeup and why that matters for
the division-timing result below.*

### Visualize one frame with annotated cell centers

In [ ]:
frame = np.asarray(ds.image[FRAME_INDEX])
mip = frame.max(axis=0)  # max-intensity projection over Z

node_attrs = ds.tracks.node_attrs()
frame_nodes = node_attrs.filter(node_attrs["t"] == FRAME_INDEX)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(mip, cmap="viridis")
ax.scatter(frame_nodes["x"], frame_nodes["y"], s=12, facecolors="none", edgecolors="white", linewidths=0.8)
ax.set_title(f"{name} — frame {FRAME_INDEX} — {frame_nodes.height} annotated cells")
ax.axis("off")
fig.savefig(OUTPUT_DIR / "frame_viz.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {OUTPUT_DIR / 'frame_viz.png'}")

*Insight: confirmed on the actual rendered image — see
`docs/2_eda_insights.md` for the embedded PNG. The one annotated center
does land on a real nucleus, not background (no scale/orientation bug),
and the image makes the sparsity finding above viscerally obvious: dozens
of clearly visible nuclei in frame 0 alone, only one annotated.*

### Animated preview: cell centers over time

In [ ]:
import imageio.v2 as imageio
from matplotlib.backends.backend_agg import FigureCanvasAgg

n_gif_frames = min(N_GIF_FRAMES, ds.image.shape[0])
gif_frames = []
for t in range(n_gif_frames):
    mip_t = np.asarray(ds.image[t]).max(axis=0)
    nodes_t = node_attrs.filter(node_attrs["t"] == t)

    fig_t, ax_t = plt.subplots(figsize=(5, 5), dpi=100)
    ax_t.imshow(mip_t, cmap="viridis")
    ax_t.scatter(nodes_t["x"], nodes_t["y"], s=25, facecolors="none", edgecolors="white", linewidths=1.2)
    ax_t.set_title(f"{name} — t={t}", color="white", fontsize=10)
    ax_t.axis("off")
    fig_t.patch.set_facecolor("black")

    canvas = FigureCanvasAgg(fig_t)
    canvas.draw()
    gif_frames.append(np.asarray(canvas.buffer_rgba()))
    plt.close(fig_t)

gif_path = OUTPUT_DIR / "track_preview.gif"
imageio.mimsave(gif_path, gif_frames, fps=4)
print(f"Saved {n_gif_frames}-frame GIF to {gif_path}")

*Insight: the animated version (embedded in `docs/2_eda_insights.md`)
shows the same pattern holding across all 30 rendered timepoints — the
annotated track moves plausibly with a real nucleus rather than jumping
around, while the overwhelming majority of visible cells stay unannotated
throughout.*

### Division timing (this dataset)

In [ ]:
edge_attrs = ds.tracks.edge_attrs()
out_degree = edge_attrs.group_by("source_id").len()
n_divisions = (out_degree["len"] == 2).sum()
print(f"division events (source with 2 outgoing edges): {n_divisions}")
print(f"total timepoints: {ds.image.shape[0]}")

*Insight: 0 divisions in `44b6_0113de3b`'s 100 timepoints — and per the
section 3/4 survey, only **1 division across all 10 surveyed videos'
combined 1,000 timepoints**. A wider sample backs this up: divisions are
genuinely rare in this dataset, not simply under-annotated in this one
(low-density) video. Consistent with `02_baseline_modeling.ipynb`'s
validation finding that the current checkpoint gets 0 division_jaccard
credit — see `docs/2_eda_insights.md`.*

## 6. Findings / limitations / next experiment

- **Findings**: 199 train videos with ground truth, only **4 test
  videos** — confirmed via section 2's `list_datasets` call against the
  real competition mount, so this is the actual held-out test set, not a
  partial slice. All 10 surveyed train videos share identical shape/scale
  — `(100, 64, 256, 256)`, `(1.625, 0.4062, 0.4062)` µm — but annotation
  density varies **~15×** across them (0.51–7.88 nodes/timepoint), and GT
  annotates only **0.13%–1.34%** of the estimated true cell count
  (`estimated_number_of_nodes`, present on 10/10 surveyed train videos).
  **That field is absent on all 4 test videos** — they ship no `.geff` at
  all, confirmed by direct read (section 4) — ruling out any
  `DET_THRESHOLD` calibration approach built around it. Only **1 division
  across 10 videos' combined 1,000 timepoints** — genuinely rare, not just
  under-annotated in one sample. Full writeup with embedded charts/images:
  `docs/2_eda_insights.md`.
- **Limitations**: `N_VIDEOS_FOR_STATS` caps the multi-video survey (default
  10, of 199 found) for a fast first pass — rerun with `None` for the full
  dataset now that the notebook's Kaggle runtime is known to be fast, to
  confirm these findings hold at full scale rather than in a 10-video
  sample.
- **Next**: since per-video count calibration isn't viable on test, the
  `DET_THRESHOLD` sweep miss (`02_baseline_modeling.ipynb`,
  `docs/4_experiments.md`) is more likely explained by the
  multiple-comparisons hypothesis than the count-budget one. Full
  prioritized roadmap:
  [`docs/3_strategy.md`](https://github.com/tuannm3812/kaggle-biohub-cell-tracking-during-development/blob/main/docs/3_strategy.md).